# Task 1: Time-Series Preprocessing and Exploratory Analysis
### Sonia's Contribution — Formative 1: Building a Pipeline for Time Series Data
### Dataset: Walmart Store Sales Forecasting

I was responsible for Task 1 of this group formative: exploratory data analysis, preprocessing, and the initial forecasting model for our time-series pipeline.

This notebook covers:
- **1A.** Understanding the dataset (time range, frequency, missing values, distributions)
- **1B.** 5 analytical questions with visualizations (including 2 using lag features and moving averages)
- **1C.** Model training with hyperparameter tuning and an experiment comparison table

The cleaned, merged dataset I produce at the end of this notebook is what I handed off to my teammates for Task 2 (database design) and Task 3 (API endpoints), so the whole group works from the same cleaned data.


In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install kaggle -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", None)


## Step 0: Get the data

I'm using the Walmart Store Sales Forecasting dataset (a public Kaggle dataset mirror of the same train/features/stores files), pulled in through the Kaggle API using a saved token rather than the older `kaggle.json` upload flow:

1. Generated a Kaggle API token from Account → API Tokens
2. Saved it as a Colab secret named `KAGGLE_TOKEN` (keeps it out of the notebook itself)
3. Wrote it to `/root/.kaggle/access_token` and ran the download


In [ ]:
# Load my Kaggle token from Colab secrets and save it where the kaggle CLI expects it
from google.colab import userdata
import os
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(userdata.get("KAGGLE_TOKEN"))
os.chmod("/root/.kaggle/access_token", 0o600)
print("Token saved")

In [ ]:
# Downloading the dataset
!kaggle datasets download -d aslanahmedov/walmart-sales-forecast
!unzip -o walmart-sales-forecast.zip -d walmart_data
!ls walmart_data


In [ ]:
# ============================================================
# LOAD THE DATA
# ============================================================
train = pd.read_csv("walmart_data/train.csv", parse_dates=["Date"])
features = pd.read_csv("walmart_data/features.csv", parse_dates=["Date"])
stores = pd.read_csv("walmart_data/stores.csv")

print("train:", train.shape)
print("features:", features.shape)
print("stores:", stores.shape)
train.head()


In [ ]:
# Merge into a single analysis-ready dataframe
df = train.merge(features, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
df = df.merge(stores, on="Store", how="left")

# features.csv has an IsHoliday column too (duplicate) - drop it, keep train's version
if "IsHoliday_feat" in df.columns:
    df = df.drop(columns=["IsHoliday_feat"])

df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
print(df.shape)
df.head()


## 1A. Understanding the Dataset

In [ ]:
# --- Time range & frequency ---
print("Earliest date:", df["Date"].min())
print("Latest date:", df["Date"].max())
print("Number of unique dates:", df["Date"].nunique())

date_diffs = df["Date"].drop_duplicates().sort_values().diff().dropna()
print("Most common gap between records:", date_diffs.mode()[0])


**Findings:** the dataset spans **2010-02-05 to 2012-10-26** (~143 weeks) at a **weekly granularity** — one row per Store + Department + Date combination.

In [ ]:
# --- Missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_summary


**Handling missing values:**
- `MarkDown1`–`MarkDown5`: these promotional markdown columns are mostly missing because markdowns didn't exist/weren't recorded for most of the study period (only introduced Nov 2011). Missing here genuinely means **"no markdown ran"**, so we fill with `0` rather than imputing a statistic — imputing a mean would fabricate a promotion that didn't happen.
- `CPI` and `Unemployment`: these are slow-moving macroeconomic indicators reported periodically. Missing values are filled with **forward-fill within each store**, since the true value doesn't change much week to week and forward-filling preserves the trend better than a global mean.


In [ ]:
df[["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]] = df[
    ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
].fillna(0)

df = df.sort_values(["Store", "Date"])
df["CPI"] = df.groupby("Store")["CPI"].ffill().bfill()
df["Unemployment"] = df.groupby("Store")["Unemployment"].ffill().bfill()

print("Remaining missing values:\n", df.isnull().sum().sum())


In [ ]:
# --- Statistical distribution of numerical columns ---
num_cols = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment", "Size"]
df[num_cols].describe()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()


**Findings:** `Weekly_Sales` is heavily right-skewed with a small number of very large sales weeks (holiday spikes). `Temperature`, `Fuel_Price`, `CPI`, and `Unemployment` are roughly continuous and cover the full range expected for US regions 2010–2012. `Size` is multimodal, reflecting the 3 discrete store types (A, B, C).

## 1B. Analytical Questions

We aggregate to **total weekly sales across all stores/departments** for the trend-level questions, since that's the cleanest view of the overall time series.

In [ ]:
weekly_total = df.groupby("Date").agg(
    Weekly_Sales=("Weekly_Sales", "sum"),
    IsHoliday=("IsHoliday", "max"),
    Temperature=("Temperature", "mean"),
    Fuel_Price=("Fuel_Price", "mean"),
    CPI=("CPI", "mean"),
    Unemployment=("Unemployment", "mean"),
).reset_index().sort_values("Date").reset_index(drop=True)

weekly_total.head()


### Q1. Does total weekly sales have an increasing/decreasing trend over the study period?

In [ ]:
plt.figure()
plt.plot(weekly_total["Date"], weekly_total["Weekly_Sales"])
plt.title("Total Weekly Sales Over Time (2010-2012)")
plt.xlabel("Date")
plt.ylabel("Total Weekly Sales ($)")
plt.show()


**Interpretation:** sales show sharp seasonal spikes around November/December (Thanksgiving + Christmas) each year rather than a steady long-term trend. Outside the holiday season, sales fluctuate around a fairly stable baseline — the dominant pattern here is **seasonality, not a secular trend**.

### Q2. Do holiday weeks have significantly higher sales than non-holiday weeks?

In [ ]:
holiday_avg = df.groupby("IsHoliday")["Weekly_Sales"].mean()
print(holiday_avg)

plt.figure()
sns.barplot(x=holiday_avg.index.map({True: "Holiday", False: "Non-Holiday"}), y=holiday_avg.values)
plt.title("Average Weekly Sales: Holiday vs Non-Holiday")
plt.ylabel("Average Weekly Sales ($)")
plt.show()


**Interpretation:** holiday weeks average noticeably higher sales per store-department than non-holiday weeks, confirming the spikes seen in Q1 are holiday-driven.

### Q3. Does store type/size correlate with average weekly sales?

In [ ]:
type_sales = df.groupby("Type")["Weekly_Sales"].mean().sort_values(ascending=False)
print(type_sales)

plt.figure()
sns.scatterplot(data=df.groupby("Store").agg({"Size": "first", "Weekly_Sales": "mean"}).reset_index(),
                 x="Size", y="Weekly_Sales")
plt.title("Store Size vs Average Weekly Sales")
plt.xlabel("Store Size (sq ft)")
plt.ylabel("Average Weekly Sales ($)")
plt.show()


**Interpretation:** larger stores (Type A) tend to have higher average weekly sales than smaller Type C stores, and there's a visible positive relationship between store size and average sales.

### Q4. Do external economic variables (Temperature, Fuel Price, CPI, Unemployment) correlate with sales over time?

In [ ]:
corr_cols = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
corr_matrix = weekly_total[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation: Weekly Sales vs External Variables")
plt.show()


**Interpretation:** correlations between weekly sales and Temperature, Fuel Price, CPI, and Unemployment are weak in this weekly, store-aggregated view — consistent with sales being driven far more by calendar/seasonal effects (holidays) than by these macro variables.

### Q5 (lag features). Is this week's total sales correlated with sales from the previous 1-4 weeks?

In [ ]:
for lag in [1, 2, 3, 4]:
    weekly_total[f"lag_{lag}"] = weekly_total["Weekly_Sales"].shift(lag)

lag_corr = weekly_total[["Weekly_Sales", "lag_1", "lag_2", "lag_3", "lag_4"]].corr()["Weekly_Sales"]
print(lag_corr)

plt.figure()
plt.bar([f"lag_{i}" for i in range(1, 5)], lag_corr.iloc[1:])
plt.title("Correlation of Weekly Sales with Lagged Sales")
plt.ylabel("Correlation")
plt.show()


**Interpretation:** sales are strongly autocorrelated at lag 1 (last week's sales are a strong predictor of this week's), with correlation weakening at longer lags — a classic sign that recent history carries real predictive signal, justifying lag features in our model.

### Q6 (moving average). Does a 4-week moving average reveal the underlying trend more clearly than raw weekly sales?

In [ ]:
weekly_total["MA_4"] = weekly_total["Weekly_Sales"].shift(1).rolling(window=4).mean()
weekly_total["MA_12"] = weekly_total["Weekly_Sales"].shift(1).rolling(window=12).mean()

plt.figure()
plt.plot(weekly_total["Date"], weekly_total["Weekly_Sales"], alpha=0.4, label="Raw Weekly Sales")
plt.plot(weekly_total["Date"], weekly_total["MA_4"], label="4-week Moving Average")
plt.plot(weekly_total["Date"], weekly_total["MA_12"], label="12-week Moving Average")
plt.legend()
plt.title("Raw Sales vs Moving Averages")
plt.show()


**Interpretation:** the 4-week moving average smooths out weekly noise while still capturing the holiday spikes, whereas the 12-week moving average reveals a smoother seasonal wave — useful for the model as a stabilized feature rather than the raw, noisy weekly value.

## 1C. Model Training

I initially modeled the **national weekly total** (aggregating all 45 stores into one series), but with only ~131 weekly points that's a small-sample regression, and it collapses away the store-level differences my own Q3 finding showed (Type A vs Type C stores have very different sales levels). Instead, I model **weekly sales per store**, which keeps thousands of rows and lets Store size/type act as real predictive features. I predict **per-store weekly sales** using lagged sales, moving averages (computed within each store's own history), store size/type, and calendar/economic features. I compare a **Linear Regression baseline** against a **tuned Random Forest**, and run a small hyperparameter search as my 2+ experiments.


In [ ]:
# ============================================================
# FEATURE ENGINEERING FOR MODELING (per-store, not national aggregate)
# ============================================================
# Aggregate to Store + Date level (summing across departments within a store)
store_weekly = df.groupby(["Store", "Date"]).agg(
    Weekly_Sales=("Weekly_Sales", "sum"),
    IsHoliday=("IsHoliday", "max"),
    Temperature=("Temperature", "mean"),
    Fuel_Price=("Fuel_Price", "mean"),
    CPI=("CPI", "mean"),
    Unemployment=("Unemployment", "mean"),
    Type=("Type", "first"),
    Size=("Size", "first"),
).reset_index()

store_weekly = store_weekly.sort_values(["Store", "Date"]).reset_index(drop=True)

# Lag and moving-average features computed WITHIN each store's own history
# (shift(1) before rolling avoids the leakage bug my teammate flagged earlier)
g = store_weekly.groupby("Store")["Weekly_Sales"]
for lag in [1, 2, 3, 4]:
    store_weekly[f"lag_{lag}"] = g.shift(lag)
store_weekly["MA_4"] = g.transform(lambda s: s.shift(1).rolling(window=4).mean())
store_weekly["MA_12"] = g.transform(lambda s: s.shift(1).rolling(window=12).mean())

model_df = store_weekly.copy()
model_df["month"] = model_df["Date"].dt.month
model_df["week_of_year"] = model_df["Date"].dt.isocalendar().week.astype(int)
model_df["IsHoliday"] = model_df["IsHoliday"].astype(int)

# One-hot encode store Type (A/B/C) instead of raw Store ID, to avoid 45 sparse dummy columns
model_df = pd.get_dummies(model_df, columns=["Type"], prefix="Type")
type_dummy_cols = [c for c in model_df.columns if c.startswith("Type_")]

model_df = model_df.dropna().reset_index(drop=True)  # drop rows with NaN lag/MA at the start of each store's history

feature_cols = ["lag_1", "lag_2", "lag_3", "lag_4", "MA_4", "MA_12",
                 "IsHoliday", "Temperature", "Fuel_Price", "CPI", "Unemployment",
                 "Size", "month", "week_of_year"] + type_dummy_cols
target_col = "Weekly_Sales"

X = model_df[feature_cols]
y = model_df[target_col]

# Time-based split by DATE (not row index), since many stores share the same dates -
# splitting by date keeps every store's train/test split chronologically correct
split_date = model_df["Date"].quantile(0.8, interpolation="nearest")
train_mask = model_df["Date"] < split_date
test_mask = model_df["Date"] >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# Sort the training rows by Date (not Store) so TimeSeriesSplit inside GridSearchCV
# sees a proper chronological order across the multiple stores per date
train_order = model_df.loc[train_mask, "Date"].sort_values().index
X_train, y_train = X_train.loc[train_order], y_train.loc[train_order]

print("Split date:", split_date.date())
print("Train size:", X_train.shape, "Test size:", X_test.shape)


In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "Experiment": name,
        "MAE": round(mean_absolute_error(y_true, y_pred), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        "R2": round(r2_score(y_true, y_pred), 4),
    }

results = []


### Experiment 1: Linear Regression (baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
results.append(evaluate("Exp 1: Linear Regression (baseline)", y_test, lr_preds))
results[-1]


### Experiment 2: Random Forest with hyperparameter tuning (GridSearchCV)

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 2],
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)

best_rf = grid_search.best_estimator_
rf_preds = best_rf.predict(X_test)
results.append(evaluate("Exp 2: Random Forest (tuned via GridSearchCV)", y_test, rf_preds))
results[-1]


### Experiment 3: Random Forest, default hyperparameters (for direct tuning comparison)

In [ ]:
rf_default = RandomForestRegressor(n_estimators=100, random_state=42)
rf_default.fit(X_train, y_train)
rf_default_preds = rf_default.predict(X_test)
results.append(evaluate("Exp 3: Random Forest (default params)", y_test, rf_default_preds))
results[-1]


### Experiment comparison table

In [ ]:
experiment_table = pd.DataFrame(results)
experiment_table


In [ ]:
# With per-store test rows, aggregate to one national total per date so the chart stays readable
plot_df = model_df.loc[test_mask, ["Date"]].copy()
plot_df["Actual"] = y_test.values
plot_df["Predicted"] = rf_preds
plot_agg = plot_df.groupby("Date")[["Actual", "Predicted"]].sum().reset_index()

plt.figure()
plt.plot(plot_agg["Date"], plot_agg["Actual"], label="Actual", linewidth=2)
plt.plot(plot_agg["Date"], plot_agg["Predicted"], label="Predicted (Tuned RF)", linestyle="--")
plt.legend()
plt.title("Actual vs Predicted Weekly Sales, Summed Across Stores (Test Period)")
plt.xlabel("Date")
plt.ylabel("Weekly Sales ($)")
plt.show()


**Interpretation:** with the moving-average leakage fixed, these results reflect genuine forecasting performance rather than a model exploiting information it shouldn't have access to. Compare the MAE/RMSE/R² across the three experiments above and note which model actually performs best here — with the leak removed, Linear Regression should no longer show an unrealistic perfect score, and the Random Forest tuned-vs-default comparison becomes a fair test of whether hyperparameter tuning helped.

I could add an LSTM as a 4th experiment if a deep learning comparison would strengthen the report — the `X_train`/`y_train` arrays would just need reshaping to `(samples, timesteps, features)`.


## Exporting my cleaned dataset for the team

This is the final output of my Task 1 work — a cleaned, merged version of the raw data with missing values handled. I'm exporting it here so I can send it to my teammates working on Task 2 (database design) and Task 3 (API), so we're all building on the same cleaned data instead of everyone re-doing the cleaning step separately.

In [ ]:
df.to_csv("walmart_cleaned_merged.csv", index=False)
weekly_total.to_csv("walmart_weekly_totals_with_features.csv", index=False)

from google.colab import files
files.download("walmart_cleaned_merged.csv")
files.download("walmart_weekly_totals_with_features.csv")
print("My cleaned Task 1 dataset is ready - sending this to the team for Task 2 and Task 3.")
